<a href="https://colab.research.google.com/github/sambslam/PitchXAI-Assignment/blob/main/Task3_Browser_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3 — Autonomous Browser Agent (browser-use + Ollama 8B on RunPod)

**PitchX assignment — Task 3**

This notebook documents an autonomous web-browsing agent built with [browser-use](https://github.com/browser-use/browser-use) driven by a **local Llama 3.1 8B** model served through **Ollama**, running on a **RunPod** GPU pod (NVIDIA RTX 6000 Ada, 48 GB VRAM).

> **How to read this notebook.** The agent runs on the RunPod pod, where Ollama serves the model on `localhost:11434`. Google Colab is a separate machine and cannot reach the pod's `localhost`, so the cells here are **documentation of what was run on the pod**, not live-executable in Colab. During the technical interview, the same commands and script can be executed directly on the provided RunPod access. Real captured logs from the pod are included below.

---


## 1. Architecture — how the agent works

A language model on its own can only read and write text. It cannot click a button or read a live web page. To make it *act* on the web, you pair it with a tool that gives it eyes and hands and run a loop. That is what browser-use provides.

**The two pieces:**

- **browser-use (eyes + hands).** A Python library that drives a real Chromium browser via Playwright. It reads the current page and turns it into a text/accessibility description the model can understand (the *eyes*), and it executes the actions the model decides on — click, scroll, type, extract (the *hands*).
- **Ollama running Llama 3.1 8B (the brain).** Ollama serves the model locally on the pod. The model receives the page description plus the goal and decides the next action.

**The loop (observe → decide → act):**

1. browser-use loads the page and builds a text description of it.
2. That description + the task goal is sent to the Ollama model.
3. The model replies with the next action (e.g. *extract the H1*, *scroll*, *done*).
4. browser-use executes that action in the real browser.
5. The page state changes; browser-use re-reads it.
6. Repeat until the model calls `done` (or a step cap is hit).

This observe–decide–act cycle is the core of any agent; browser-use specializes it for the web.

**Why this maps to PitchX.** PitchX automates lead research and enrichment. A browser agent that can visit a company page and pull specific facts ("what does this company do", "when was it founded") is exactly the kind of automated enrichment step that feeds outreach.


## 2. Environment

| Component | Value |
|---|---|
| Platform | RunPod GPU Pod (On-Demand) |
| GPU | NVIDIA RTX 6000 Ada Generation, 48 GB VRAM |
| Driver / CUDA | 550.127.05 / CUDA 12.4 |
| Template | PyTorch (CUDA 12.x) |
| Container disk / Volume | 30 GB / 50 GB (models stored on persistent volume) |
| Python | 3.12.3 |
| Model server | Ollama |
| Model | `llama3.1:8b` (text-only) |
| Agent framework | browser-use 0.13.1 |

> **Note on the "fine-tune" wording.** The task says *"fine tune it with browser use so it works as auto browser."* In practice browser-use works with a **stock** Ollama model out of the box — no weight fine-tuning is required to get a functioning browser agent, and that is the deliverable shown here. True weight fine-tuning is a separate, much heavier process and is unnecessary for browser control. This is documented as an honest interpretation of an ambiguously worded requirement.


## 3. Setup (run on the RunPod pod terminal)

These commands were run in the pod's web terminal, **not** in Colab.

### 3.1 Point Ollama's model storage at the persistent volume
By default Ollama stores models on the container disk, which is wiped when the pod is terminated. Pointing `OLLAMA_MODELS` at the volume (`/workspace`) means the model survives stop/restart and does not need re-downloading.


In [ ]:
# --- pod terminal ---
echo 'export OLLAMA_MODELS=/workspace/ollama_models' >> ~/.bashrc
source ~/.bashrc
mkdir -p /workspace/ollama_models

### 3.2 Install Ollama, start the server, pull the model
The install warns about `systemd not running` (expected inside a container — we start the server manually) and about GPU detection (a false alarm caused by a missing `lspci` tool; `nvidia-smi` confirms the GPU is present).


In [ ]:
# --- pod terminal ---
curl -fsSL https://ollama.com/install.sh | sh

# start the server in the background, logging to the volume
ollama serve > /workspace/ollama.log 2>&1 &

# download the 8B model (~4.7 GB, saved to the volume)
ollama pull llama3.1:8b

# sanity check that the model runs on the GPU
ollama run llama3.1:8b "Say hello in one sentence."

### 3.3 Install browser-use + Playwright + Chromium
`playwright` did not come in automatically as a browser-use dependency in this version, so it is installed explicitly. The `playwright` CLI was not on PATH, so it is invoked via `python3 -m playwright`.


In [ ]:
# --- pod terminal ---
pip install browser-use
pip install playwright
python3 -m playwright install chromium
python3 -m playwright install-deps chromium

## 4. The agent script

`browser_agent.py` (stored on the pod at `/workspace/browser_agent.py`).

**Key configuration choices, and why:**

- `host="http://localhost:11434"` — connects to the Ollama server running on the same pod.
- `use_vision=False` — Llama 3.1 8B is **text-only**. By default browser-use sends page **screenshots** to the model; a text-only model rejects them with a `multimodal not supported` 400 error. Disabling vision sends only the text/accessibility representation of the page, which the model handles, and it also runs faster.
- `max_steps=10` — a hard safety cap. Without it, a small model that fails to recognise task completion can loop indefinitely (an early run reached 112 steps). The cap bounds time and GPU cost.
- An **explicit, ordered task prompt with a clear stop instruction** — small models loop when "done" is ambiguous, so the prompt spells out exactly when to stop.


In [ ]:
import asyncio
from browser_use import Agent
from browser_use.llm import ChatOllama


async def main():
    # Connect browser-use to the local Ollama model running on the pod
    llm = ChatOllama(
        model="llama3.1:8b",
        host="http://localhost:11434",
    )

    agent = Agent(
        task=(
            "Navigate to https://en.wikipedia.org/wiki/OpenAI. Read the page. "
            "In one sentence, describe what OpenAI does. Once you have this, "
            "immediately mark the task done and stop."
        ),
        llm=llm,
        use_vision=False,   # text-only model: do not send screenshots
    )

    result = await agent.run(max_steps=10)   # hard cap to prevent runaway loops
    print("\n\n=== AGENT RESULT ===")
    print(result)


if __name__ == "__main__":
    asyncio.run(main())


**Run it on the pod** (the `tee` pattern saves the log to the volume so output survives a dropped terminal):

```bash
python3 browser_agent.py 2>&1 | tee /workspace/task3_run.log
```


## 5. Results — three documented runs

The agent was evaluated on three tasks of increasing difficulty. Together they show a working agent **and** the characteristic limitations of using a small (8B) local model as the agent brain.


### Run 1 — Controlled baseline (`example.com`)

**Task:** report the main H1 heading.
**Outcome:** clean success in **2 steps**.

```
Step 1: navigate -> example.com; extract H1 -> "Example Domain"
Step 2: done (success=True)

Final Result: The main H1 heading is 'Example Domain'.
✅ Task completed successfully
```

This validates the full pipeline end-to-end (Ollama connection, Chromium launch, page read, observe–decide–act loop, clean termination) on a minimal, static page where the model cannot get confused.


### Run 2 — Realistic page, two-part task (OpenAI Wikipedia, founding year + what it does)

**Task:** report what the company does **and** the year founded.
**Outcome:** success in **2 steps**, but the model answered **only one part** — the founding year — and dropped the "what it does" half.

```
Step 1: extract "Year OpenAI was founded" -> "OpenAI was founded in 2015."
Step 2: done (success=True)

Final Result: OpenAI was founded in 2015
```

**Observation (limitation).** The model's own step-1 memory noted "OpenAI, a company focused on artificial intelligence research," so it had registered what the company does — but it only ran an `extract` for the year, then declared done. **Small models do not reliably track multi-part instructions to completion.**


### Run 3 — Realistic page, single fact (OpenAI Wikipedia, what it does)

**Task:** in one sentence, describe what OpenAI does.
**Outcome:** success in **7 steps**, after some inefficient wandering and a self-correction.

```
Step 1-3: tried search_page for literal "OpenAI does" / "what OpenAI does"
          / "mission statement" -> 0 matches (those exact phrases aren't on the page)
          (one step hit a 180s timeout)
Step 4-5: switched to the `extract` tool (semantic, not literal text match)
          -> found the mission statement
Step 6-7: done (success=True)

Final Result: The mission statement of OpenAI is to ensure that artificial
general intelligence benefits all of humanity.
```

**Observation (limitation).** The model first reached for literal text matching, which failed because the page does not contain those exact phrases, then **self-corrected** to semantic extraction and succeeded. A larger model would typically go straight to extraction. This illustrates the inefficient reasoning paths small models take — capable, but not direct.


## 6. Findings and limitations

**What works well**
- The full agent pipeline is solid: local 8B model + browser-use + real Chromium completing autonomous web tasks on a RunPod GPU.
- On simple, well-scoped tasks the agent is fast and reliable (Run 1).
- With an explicit prompt and a step cap, it completes realistic extraction tasks (Run 3).

**Limitations of an 8B local model as the agent brain**
- **Multi-part instructions** are unreliable — it may complete only one sub-goal then stop (Run 2).
- **Inefficient reasoning paths** — it can try approaches that don't fit (literal search vs. semantic extraction) before self-correcting (Run 3).
- **Runaway loops** without a `max_steps` cap when task completion is ambiguous (early run reached 112 steps).

**Mitigations applied**
- `use_vision=False` to match the text-only model.
- Explicit, ordered task prompts with a clear stop condition.
- `max_steps=10` as a hard safety cap.

**Possible improvements (documented, not implemented here)**
- Use a **vision-capable** model (e.g. `llama3.2-vision`) to let the agent see screenshots for visually complex pages.
- Use a **larger or more agent-tuned** model — the 48 GB RTX 6000 Ada has ample headroom beyond 8B — to improve multi-step reliability and reduce wandering.
- Add a `fallback_llm` so a single provider error does not stall the run.


## 7. Reproduction summary

1. Deploy a RunPod GPU pod (PyTorch / CUDA 12.x template), 30 GB container / 50 GB volume.
2. Set `OLLAMA_MODELS=/workspace/ollama_models`, install Ollama, start `ollama serve`, `ollama pull llama3.1:8b`.
3. `pip install browser-use playwright` and `python3 -m playwright install chromium` (+ `install-deps`).
4. Save `browser_agent.py` to `/workspace`, run with `python3 browser_agent.py`.
5. The agent navigates, reads the page, reasons via the local model, and reports the result.

*This notebook is documentation; live execution is performed on RunPod.*
